In [ ]:
import cv2
import numpy as np
import mediapipe as mp
from mediapipe.tasks.python import vision
from mediapipe.tasks.python.core.base_options import BaseOptions
from src.feature_extraction import FeatureExtractor

f=FeatureExtractor()

YAW_THRESHOLD   = 20.0   # left / right
PITCH_THRESHOLD = 15.0   # up / down
ROLL_THRESHOLD  = 15.0   # tilt (optional)

# -----------------------------
# Webcam
# -----------------------------

cap = cv2.VideoCapture(0)

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break
    frame=frame.copy() # avoid modifying the original frame
    model_image=cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    h, w = frame.shape[:2]
    
    features=f._extract_head_pose(model_image,w,h)

    if features:
        rvec=features["rvec"]
        tvec=features["tvec"]
        cam_matrix=features["camera_matrix"]
        dist_coeffs=features["dist_coeffs"]
        nose_2d=features["nose_2d"]
        pitch=features["head_pitch"]
        yaw=features["head_yaw"]
        roll=features["head_roll"]
        head_pose=features["head_pose"]
        
        cv2.putText(frame, f"Pitch: {pitch:.1f}", (20, 40),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.putText(frame, f"Yaw: {yaw:.1f}", (20, 70),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.putText(frame, f"Roll: {roll:.1f}", (20, 100),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)
        cv2.putText(frame, f"Head Pose: {head_pose}", (20, 130),cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0,255,0), 2)

        cv2.putText(frame, f"yaw,pitch,roll in radians: {yaw:.2f}, {pitch :.2f}, {roll:.2f}", (20, 160),cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0,255,0), 2)
        
        axis = np.float64([
            [50, 0, 0],
            [0, 50, 0],
            [0, 0, 50] ]
            )
        
        imgpts, _ = cv2.projectPoints( axis, rvec, tvec, cam_matrix, dist_coeffs )
        nose = tuple(map(int, nose_2d))
        print("frame id:", id(frame), "nose:", nose_2d)

        cv2.line(frame, nose, tuple(imgpts[0].ravel().astype(int)), (0,0,255), 3)
        cv2.line(frame, nose, tuple(imgpts[1].ravel().astype(int)), (0,255,0), 3)
        cv2.line(frame, nose, tuple(imgpts[2].ravel().astype(int)), (255,0,0), 3)
        
    cv2.imshow("Head Pose Estimation", frame)
    if cv2.waitKey(1) & 0xFF == 27: # ESC key to exit
        break

cap.release()
cv2.destroyAllWindows()

In [ ]:
from ultralytics import YOLO
model = YOLO("yolo26n.pt")  # This downloads the YOLO26n weights for you

In [3]:
import cv2
import numpy as np
from src.feature_extraction import FeatureExtractor

# Initialize extractor
extractor = FeatureExtractor()

def visualize_gaze(frame_bgr):
    h, w = frame_bgr.shape[:2]

    # MediaPipe expects RGB
    frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
    features = extractor._extract_gaze_features(frame_rgb, w, h)

    # ---------- Draw pupils ----------
    if features["pupil_left_x"] > 0:
        cv2.circle(frame_bgr, (int(features["pupil_left_x"]), int(features["pupil_left_y"])), 3, (0, 255, 0), -1 )

    if features["pupil_right_x"] > 0:
        cv2.circle(
            frame_bgr,
            (int(features["pupil_right_x"]), int(features["pupil_right_y"])),
            3,
            (0, 255, 0),
            -1
        )

    # ---------- Draw gaze point ----------
    gaze_x = int(features["gazePoint_x"])
    gaze_y = int(features["gazePoint_y"])

    if gaze_x > 0:
        cv2.circle(frame_bgr, (gaze_x, gaze_y), 5, (255, 0, 0), -1)

        # Draw arrow from face center → gaze point
        face_center = (w // 2, h // 2)
        cv2.arrowedLine(
            frame_bgr,
            face_center,
            (gaze_x, gaze_y),
            (0, 0, 255),
            2,
            tipLength=0.2
        )

    # ---------- Text info ----------
    cv2.putText(
        frame_bgr,
        f"Gaze: {features['gaze_direction']}",
        (20, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )

    cv2.putText(
        frame_bgr,
        f"Gaze_directions: {features['dx']:.2f}, {features['dy']:.2f}",
        (40, 30),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (255, 255, 255),
        2
    )
    
    cv2.putText(
        frame_bgr,
        f"On Script: {features['gaze_on_script']}",
        (20, 60),
        cv2.FONT_HERSHEY_SIMPLEX,
        0.7,
        (0, 255, 255),
        2
    )
    return frame_bgr

In [4]:
cap = cv2.VideoCapture(0)
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    frame = visualize_gaze(frame)
    cv2.imshow("Gaze Visualization", frame)

    if cv2.waitKey(1) & 0xFF == 27:  # ESC
        break
    
cap.release()
cv2.destroyAllWindows()

Gaze dx, dy: -141.26870379144958 304.84660014229985
Gaze dx, dy: -74.39320563806547 348.42285414194794
Gaze dx, dy: -62.94736750450306 311.8157029680251
Gaze dx, dy: -77.92066567210932 310.5926117613759
Gaze dx, dy: -122.78763643093612 292.8885770050447
Gaze dx, dy: -108.00635661138352 288.23909242282036
Gaze dx, dy: -89.39394120952704 297.96275665831854
Gaze dx, dy: -81.03874167690435 312.70906166992904
Gaze dx, dy: -92.85604204607046 289.79493589941194
Gaze dx, dy: -99.80142497717159 296.72630867784505
Gaze dx, dy: -73.45194790923716 309.00448068375795
Gaze dx, dy: -114.88219583280409 289.63126506017034
Gaze dx, dy: -103.48245994657933 279.89687875122377
Gaze dx, dy: -128.36872845483384 292.7295983173394
Gaze dx, dy: -81.63043531344948 303.55988615402475
Gaze dx, dy: -119.67710662031968 298.02135737471986
Gaze dx, dy: -97.18586696080996 294.9672092460514
Gaze dx, dy: -108.38726377393064 278.6466109789134
Gaze dx, dy: -54.88365307228315 305.3641106438529
Gaze dx, dy: -76.5724674711475